# Confidence別の探索分析

歴史的な実験コードです。現在の実行入口は `../09_confidence_nested.ipynb`。
元Notebookのセル番号は0始まりです。コードの個人フォルダ名は置換しています。独立実行は保証しません。
保存出力は `../../results/imported_20260907/`、監査は `../../docs/CONFIDENCE_AUDIT.md` を参照してください。


## 元のセル index 33


In [ ]:
# ============================================================
# 15分足10年
# OOS CONFIDENCE vs PROFITABILITY
#
# 目的
# ------------------------------------------------------------
# RandomForestの予測Confidenceが高いほど
#
# 1. Direction Accuracyが高くなるか
# 2. 平均Returnが改善するか
# 3. Profit Factorが改善するか
# 4. 取引コストを超えるEdgeがあるか
#
# を完全Out-of-Sampleで検証する。
#
# 今回は
# ・TP/SLなし
# ・BET sizingなし
# ・Testで閾値最適化なし
#
# まず「Confidenceに意味があるか」だけを見る。
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


# ============================================================
# 1. 設定
# ============================================================

CSV_PATH = (
    Path.cwd()
    / "dukascopy_usdjpy"
    / "usdjpy_15m_2016_2026.csv"
)

# 15分足2本 = 30分先
HORIZON_BARS = 2

N_FOLDS = 5

RANDOM_STATE = 42

# 往復コスト仮定
COST_ZERO = 0.0
COST_REALISTIC = 0.00004   # 0.004%

# Confidence threshold比較
CONF_THRESHOLDS = [
    0.50,
    0.52,
    0.54,
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
    0.70,
    0.75,
]

# Bootstrap
N_BOOTSTRAP = 5000


# ============================================================
# 2. データ読み込み
# ============================================================

df = pd.read_csv(
    CSV_PATH,
    index_col=0,
    parse_dates=True
)

df.index = pd.to_datetime(
    df.index,
    utc=True
)

df = (
    df
    .sort_index()
    .copy()
)

df.columns = [
    c.lower()
    for c in df.columns
]


print(
    "データ読み込み完了"
)

print(
    "総行数:",
    len(df)
)

print(
    "期間:",
    df.index.min(),
    "→",
    df.index.max()
)


# ============================================================
# 3. RSI
# ============================================================

def calculate_rsi(
    close,
    period=14
):

    delta = close.diff()

    gain = delta.clip(
        lower=0
    )

    loss = -delta.clip(
        upper=0
    )

    avg_gain = (
        gain
        .rolling(period)
        .mean()
    )

    avg_loss = (
        loss
        .rolling(period)
        .mean()
    )

    rs = (
        avg_gain
        /
        avg_loss.replace(
            0,
            np.nan
        )
    )

    return (
        100
        -
        100
        /
        (
            1 + rs
        )
    )


# ============================================================
# 4. 特徴量
# ============================================================

def make_features(
    data
):

    x = data.copy()

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    x["return_1"] = (
        x["close"]
        .pct_change(1)
    )

    x["return_2"] = (
        x["close"]
        .pct_change(2)
    )

    x["return_4"] = (
        x["close"]
        .pct_change(4)
    )

    x["return_8"] = (
        x["close"]
        .pct_change(8)
    )

    x["return_16"] = (
        x["close"]
        .pct_change(16)
    )

    # --------------------------------------------------------
    # Volatility
    # --------------------------------------------------------

    x["vol_4"] = (
        x["return_1"]
        .rolling(4)
        .std()
    )

    x["vol_8"] = (
        x["return_1"]
        .rolling(8)
        .std()
    )

    x["vol_16"] = (
        x["return_1"]
        .rolling(16)
        .std()
    )

    x["vol_32"] = (
        x["return_1"]
        .rolling(32)
        .std()
    )

    # --------------------------------------------------------
    # MA
    # --------------------------------------------------------

    for period in [
        5,
        10,
        20,
        50,
        100,
    ]:

        ma = (
            x["close"]
            .rolling(period)
            .mean()
        )

        x[
            f"ma{period}_distance"
        ] = (
            x["close"]
            /
            ma
            - 1
        )

        x[
            f"ma{period}_slope"
        ] = (
            ma.pct_change()
        )

    # --------------------------------------------------------
    # Candle
    # --------------------------------------------------------

    candle_range = (
        x["high"]
        -
        x["low"]
    ).replace(
        0,
        np.nan
    )

    x["body"] = (
        x["close"]
        -
        x["open"]
    ) / candle_range

    x["upper_wick"] = (
        x["high"]
        -
        x[
            [
                "open",
                "close",
            ]
        ].max(
            axis=1
        )
    ) / candle_range

    x["lower_wick"] = (
        x[
            [
                "open",
                "close",
            ]
        ].min(
            axis=1
        )
        -
        x["low"]
    ) / candle_range

    x["range_pct"] = (
        x["high"]
        -
        x["low"]
    ) / x["close"]

    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    x["rsi14"] = (
        calculate_rsi(
            x["close"],
            14
        )
        /
        100
    )

    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------

    previous_close = (
        x["close"]
        .shift(1)
    )

    true_range = pd.concat(
        [
            x["high"]
            -
            x["low"],

            (
                x["high"]
                -
                previous_close
            ).abs(),

            (
                x["low"]
                -
                previous_close
            ).abs(),
        ],
        axis=1
    ).max(
        axis=1
    )

    x["atr14"] = (
        true_range
        .rolling(14)
        .mean()
        /
        x["close"]
    )

    # --------------------------------------------------------
    # 高値・安値位置
    # --------------------------------------------------------

    high_16 = (
        x["high"]
        .rolling(16)
        .max()
    )

    low_16 = (
        x["low"]
        .rolling(16)
        .min()
    )

    x["distance_high_16"] = (
        high_16
        -
        x["close"]
    ) / x["close"]

    x["distance_low_16"] = (
        x["close"]
        -
        low_16
    ) / x["close"]

    # --------------------------------------------------------
    # 時刻
    # --------------------------------------------------------

    hour = (
        x.index.hour
        +
        x.index.minute
        /
        60
    )

    x["hour_sin"] = np.sin(
        2
        *
        np.pi
        *
        hour
        /
        24
    )

    x["hour_cos"] = np.cos(
        2
        *
        np.pi
        *
        hour
        /
        24
    )

    x["weekday"] = (
        x.index.dayofweek
        /
        4
    )

    return x


data = make_features(
    df
)


# ============================================================
# 5. 30分先ラベル
# ============================================================

data["future_return"] = (
    data["close"]
    .shift(
        -HORIZON_BARS
    )
    /
    data["close"]
    - 1
)

data["target"] = (
    data["future_return"]
    > 0
).astype(int)


# ============================================================
# 6. 特徴量一覧
# ============================================================

FEATURES = [

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",
    "weekday",
]


data = (
    data
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan
    )
    .dropna(
        subset=
            FEATURES
            +
            [
                "future_return",
                "target",
            ]
    )
    .copy()
)


print()

print(
    "ML使用可能データ:",
    len(data)
)


# ============================================================
# 7. Walk-Forwardで完全OOS予測を作る
# ============================================================

block = (
    len(data)
    //
    (
        N_FOLDS
        + 1
    )
)


oos_frames = []

fold_model_rows = []


for fold in range(
    1,
    N_FOLDS + 1
):

    print()

    print(
        "===================================="
    )

    print(
        f"Fold {fold}"
    )

    print(
        "===================================="
    )

    train_end = (
        block
        *
        fold
    )

    # ラベル境界のリーク防止
    test_start = (
        train_end
        +
        HORIZON_BARS
    )

    test_end = min(
        test_start
        +
        block,
        len(data)
    )

    train = (
        data.iloc[
            :train_end
        ]
        .copy()
    )

    test = (
        data.iloc[
            test_start:test_end
        ]
        .copy()
    )

    if (
        len(train) < 1000
        or
        len(test) == 0
    ):

        continue

    X_train = (
        train[
            FEATURES
        ]
    )

    y_train = (
        train[
            "target"
        ]
    )

    X_test = (
        test[
            FEATURES
        ]
    )

    y_test = (
        test[
            "target"
        ]
    )

    # --------------------------------------------------------
    # RF
    # --------------------------------------------------------

    model = RandomForestClassifier(

        n_estimators=
            300,

        max_depth=
            8,

        min_samples_leaf=
            30,

        max_features=
            "sqrt",

        class_weight=
            "balanced",

        random_state=
            RANDOM_STATE,

        n_jobs=
            -1,
    )

    model.fit(
        X_train,
        y_train
    )

    train_prob = (
        model.predict_proba(
            X_train
        )[:, 1]
    )

    test_prob = (
        model.predict_proba(
            X_test
        )[:, 1]
    )

    train_auc = (
        roc_auc_score(
            y_train,
            train_prob
        )
    )

    test_auc = (
        roc_auc_score(
            y_test,
            test_prob
        )
    )

    # --------------------------------------------------------
    # OOS保存
    # --------------------------------------------------------

    result = pd.DataFrame(
        index=
            test.index
    )

    result[
        "fold"
    ] = fold

    result[
        "p_up"
    ] = (
        test_prob
    )

    result[
        "p_down"
    ] = (
        1
        -
        test_prob
    )

    result[
        "confidence"
    ] = np.maximum(
        result[
            "p_up"
        ],
        result[
            "p_down"
        ]
    )

    result[
        "prediction"
    ] = np.where(
        result[
            "p_up"
        ]
        >=
        0.5,
        1,
        -1
    )

    result[
        "actual_up"
    ] = (
        test[
            "target"
        ].values
    )

    result[
        "future_return"
    ] = (
        test[
            "future_return"
        ].values
    )

    result[
        "correct"
    ] = np.where(
        result[
            "prediction"
        ]
        == 1,
        result[
            "future_return"
        ]
        > 0,
        result[
            "future_return"
        ]
        < 0,
    )

    # BUYならそのまま
    # SELLなら符号反転
    result[
        "gross_strategy_return"
    ] = (
        result[
            "future_return"
        ]
        *
        result[
            "prediction"
        ]
    )

    result[
        "net_return_zero_cost"
    ] = (
        result[
            "gross_strategy_return"
        ]
    )

    result[
        "net_return_cost"
    ] = (
        result[
            "gross_strategy_return"
        ]
        -
        COST_REALISTIC
    )

    result[
        "side"
    ] = np.where(
        result[
            "prediction"
        ]
        == 1,
        "BUY",
        "SELL"
    )

    oos_frames.append(
        result
    )

    fold_model_rows.append(
        {
            "fold":
                fold,

            "train_size":
                len(train),

            "test_size":
                len(test),

            "train_auc":
                train_auc,

            "test_auc":
                test_auc,

            "gap":
                train_auc
                -
                test_auc,
        }
    )

    print(
        "Train AUC:",
        round(
            train_auc,
            4
        )
    )

    print(
        "Test AUC:",
        round(
            test_auc,
            4
        )
    )


# ============================================================
# 8. OOS結合
# ============================================================

oos = (
    pd.concat(
        oos_frames
    )
    .sort_index()
)

fold_models = pd.DataFrame(
    fold_model_rows
)


print()

print(
    "総OOS行数:",
    len(oos)
)


# ============================================================
# 9. Profit Factor計算
# ============================================================

def profit_factor(
    returns
):

    r = np.asarray(
        returns,
        dtype=float
    )

    gains = (
        r[
            r > 0
        ].sum()
    )

    losses = (
        -r[
            r < 0
        ].sum()
    )

    if losses > 0:

        return (
            gains
            /
            losses
        )

    if gains > 0:

        return np.inf

    return np.nan


# ============================================================
# 10. DD / Growth
# ============================================================

def return_stats(
    returns
):

    r = np.asarray(
        returns,
        dtype=float
    )

    if len(r) == 0:

        return {
            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "median_return":
                np.nan,

            "profit_factor":
                np.nan,

            "max_dd":
                np.nan,

            "total_growth":
                np.nan,
        }

    equity = np.r_[
        1.0,
        np.cumprod(
            1 + r
        )
    ]

    peak = (
        np.maximum.accumulate(
            equity
        )
    )

    dd = (
        equity
        /
        peak
        - 1
    )

    return {
        "trades":
            len(r),

        "win_rate":
            (
                r > 0
            ).mean(),

        "avg_return":
            r.mean(),

        "median_return":
            np.median(
                r
            ),

        "profit_factor":
            profit_factor(
                r
            ),

        "max_dd":
            dd.min(),

        "total_growth":
            equity[-1]
            - 1,
    }


# ============================================================
# 11. Confidence帯
# ============================================================

CONF_BINS = [
    0.50,
    0.52,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
    0.70,
    0.75,
    0.80,
    0.90,
    1.01,
]

CONF_LABELS = [
    "50-52",
    "52-54",
    "54-56",
    "56-58",
    "58-60",
    "60-62",
    "62-65",
    "65-70",
    "70-75",
    "75-80",
    "80-90",
    "90-100",
]


oos[
    "confidence_band"
] = pd.cut(
    oos[
        "confidence"
    ],
    bins=
        CONF_BINS,
    labels=
        CONF_LABELS,
    right=False,
)


# ============================================================
# 12. Confidence帯別
# ============================================================

band_rows = []


for band in (
    CONF_LABELS
):

    subset = (
        oos.loc[
            oos[
                "confidence_band"
            ]
            ==
            band
        ]
    )

    if len(
        subset
    ) == 0:

        continue

    zero_stats = (
        return_stats(
            subset[
                "net_return_zero_cost"
            ]
        )
    )

    cost_stats = (
        return_stats(
            subset[
                "net_return_cost"
            ]
        )
    )

    band_rows.append(
        {
            "confidence_band":
                band,

            "samples":
                len(
                    subset
                ),

            "mean_confidence":
                subset[
                    "confidence"
                ].mean(),

            "direction_accuracy":
                subset[
                    "correct"
                ].mean(),

            "gross_avg_return":
                zero_stats[
                    "avg_return"
                ],

            "gross_pf":
                zero_stats[
                    "profit_factor"
                ],

            "net_avg_return":
                cost_stats[
                    "avg_return"
                ],

            "net_pf":
                cost_stats[
                    "profit_factor"
                ],
        }
    )


band_results = (
    pd.DataFrame(
        band_rows
    )
)


band_show = (
    band_results
    .copy()
)


for col in [
    "mean_confidence",
    "direction_accuracy",
    "gross_avg_return",
    "net_avg_return",
]:

    band_show[
        col
    ] *= 100


print()

print(
    "===================================="
)

print(
    "Confidence帯別"
)

print(
    "===================================="
)


print(
    band_show.to_string(
        index=False
    )
)


# ============================================================
# 13. Confidence threshold以上だけ取引
#
# これが今回特に重要
# ============================================================

threshold_rows = []


for threshold in (
    CONF_THRESHOLDS
):

    subset = (
        oos.loc[
            oos[
                "confidence"
            ]
            >=
            threshold
        ]
    )

    if len(
        subset
    ) == 0:

        continue

    gross = return_stats(
        subset[
            "net_return_zero_cost"
        ]
    )

    net = return_stats(
        subset[
            "net_return_cost"
        ]
    )

    threshold_rows.append(
        {
            "threshold":
                threshold,

            "trades":
                len(
                    subset
                ),

            "trade_rate":
                len(
                    subset
                )
                /
                len(
                    oos
                ),

            "accuracy":
                subset[
                    "correct"
                ].mean(),

            "gross_avg_return":
                gross[
                    "avg_return"
                ],

            "gross_pf":
                gross[
                    "profit_factor"
                ],

            "net_avg_return":
                net[
                    "avg_return"
                ],

            "net_pf":
                net[
                    "profit_factor"
                ],

            "net_max_dd":
                net[
                    "max_dd"
                ],

            "net_total_growth":
                net[
                    "total_growth"
                ],
        }
    )


threshold_results = (
    pd.DataFrame(
        threshold_rows
    )
)


threshold_show = (
    threshold_results
    .copy()
)


for col in [
    "threshold",
    "trade_rate",
    "accuracy",
    "gross_avg_return",
    "net_avg_return",
    "net_max_dd",
    "net_total_growth",
]:

    threshold_show[
        col
    ] *= 100


print()

print(
    "===================================="
)

print(
    "Confidence閾値以上だけ取引"
)

print(
    "===================================="
)


print(
    threshold_show.to_string(
        index=False
    )
)


# ============================================================
# 14. BUY / SELL別
# ============================================================

side_rows = []


for side in [
    "BUY",
    "SELL",
]:

    for threshold in (
        CONF_THRESHOLDS
    ):

        subset = (
            oos.loc[
                (
                    oos[
                        "side"
                    ]
                    ==
                    side
                )
                &
                (
                    oos[
                        "confidence"
                    ]
                    >=
                    threshold
                )
            ]
        )

        if len(
            subset
        ) == 0:

            continue

        gross = return_stats(
            subset[
                "net_return_zero_cost"
            ]
        )

        net = return_stats(
            subset[
                "net_return_cost"
            ]
        )

        side_rows.append(
            {
                "side":
                    side,

                "threshold":
                    threshold,

                "trades":
                    len(
                        subset
                    ),

                "accuracy":
                    subset[
                        "correct"
                    ].mean(),

                "gross_avg_return":
                    gross[
                        "avg_return"
                    ],

                "gross_pf":
                    gross[
                        "profit_factor"
                    ],

                "net_avg_return":
                    net[
                        "avg_return"
                    ],

                "net_pf":
                    net[
                        "profit_factor"
                    ],
            }
        )


side_results = (
    pd.DataFrame(
        side_rows
    )
)


side_show = (
    side_results
    .copy()
)


for col in [
    "threshold",
    "accuracy",
    "gross_avg_return",
    "net_avg_return",
]:

    side_show[
        col
    ] *= 100


print()

print(
    "===================================="
)

print(
    "BUY / SELL × Confidence"
)

print(
    "===================================="
)


print(
    side_show.to_string(
        index=False
    )
)


# ============================================================
# 15. Fold安定性
#
# 各thresholdが何Foldでプラスか
# ============================================================

fold_threshold_rows = []


for threshold in (
    CONF_THRESHOLDS
):

    for fold in sorted(
        oos[
            "fold"
        ].unique()
    ):

        subset = (
            oos.loc[
                (
                    oos[
                        "fold"
                    ]
                    ==
                    fold
                )
                &
                (
                    oos[
                        "confidence"
                    ]
                    >=
                    threshold
                )
            ]
        )

        if len(
            subset
        ) == 0:

            continue

        stats = return_stats(
            subset[
                "net_return_cost"
            ]
        )

        fold_threshold_rows.append(
            {
                "threshold":
                    threshold,

                "fold":
                    fold,

                "trades":
                    len(
                        subset
                    ),

                "avg_return":
                    stats[
                        "avg_return"
                    ],

                "pf":
                    stats[
                        "profit_factor"
                    ],
            }
        )


fold_threshold_df = (
    pd.DataFrame(
        fold_threshold_rows
    )
)


fold_stability = (
    fold_threshold_df
    .groupby(
        "threshold"
    )
    .agg(
        evaluated_folds=(
            "fold",
            "nunique"
        ),

        positive_return_folds=(
            "avg_return",
            lambda x:
                (
                    x > 0
                ).sum()
        ),

        pf_above_1_folds=(
            "pf",
            lambda x:
                (
                    x > 1
                ).sum()
        ),

        mean_fold_return=(
            "avg_return",
            "mean"
        ),

        mean_fold_pf=(
            "pf",
            "mean"
        ),
    )
    .reset_index()
)


fold_stability_show = (
    fold_stability
    .copy()
)


fold_stability_show[
    "threshold"
] *= 100


fold_stability_show[
    "mean_fold_return"
] *= 100


print()

print(
    "===================================="
)

print(
    "Confidence閾値 Fold安定性"
)

print(
    "===================================="
)


print(
    fold_stability_show.to_string(
        index=False
    )
)


# ============================================================
# 16. Bootstrap
#
# 各thresholdの平均Return 95%CI
# ============================================================

def bootstrap_mean_ci(
    values,
    n_boot=
        N_BOOTSTRAP,
    seed=
        42
):

    x = np.asarray(
        values,
        dtype=float
    )

    if len(
        x
    ) < 2:

        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
        )

    rng = (
        np.random.default_rng(
            seed
        )
    )

    bootstrap_means = np.empty(
        n_boot
    )

    for i in range(
        n_boot
    ):

        sample = rng.choice(
            x,
            size=len(x),
            replace=True
        )

        bootstrap_means[
            i
        ] = sample.mean()

    return (
        x.mean(),

        np.quantile(
            bootstrap_means,
            0.025
        ),

        np.quantile(
            bootstrap_means,
            0.975
        ),

        (
            bootstrap_means
            > 0
        ).mean(),
    )


bootstrap_rows = []


for threshold in (
    CONF_THRESHOLDS
):

    subset = (
        oos.loc[
            oos[
                "confidence"
            ]
            >=
            threshold,
            "net_return_cost"
        ]
    )

    if len(
        subset
    ) < 2:

        continue

    (
        mean_value,
        lower,
        upper,
        probability_positive,
    ) = bootstrap_mean_ci(
        subset.values
    )

    bootstrap_rows.append(
        {
            "threshold":
                threshold,

            "trades":
                len(
                    subset
                ),

            "mean_return":
                mean_value,

            "ci_2_5":
                lower,

            "ci_97_5":
                upper,

            "prob_mean_positive":
                probability_positive,
        }
    )


bootstrap_results = (
    pd.DataFrame(
        bootstrap_rows
    )
)


bootstrap_show = (
    bootstrap_results
    .copy()
)


for col in [
    "threshold",
    "mean_return",
    "ci_2_5",
    "ci_97_5",
    "prob_mean_positive",
]:

    bootstrap_show[
        col
    ] *= 100


print()

print(
    "===================================="
)

print(
    "Bootstrap 95%CI"
)

print(
    "===================================="
)


print(
    bootstrap_show.to_string(
        index=False
    )
)


# ============================================================
# 17. グラフ
# Confidence threshold vs PF
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    threshold_results[
        "threshold"
    ],
    threshold_results[
        "gross_pf"
    ],
    marker="o",
    label="Cost = 0"
)

plt.plot(
    threshold_results[
        "threshold"
    ],
    threshold_results[
        "net_pf"
    ],
    marker="o",
    label="Cost = 0.004%"
)

plt.axhline(
    1,
    linewidth=1
)

plt.xlabel(
    "Minimum confidence"
)

plt.ylabel(
    "Profit Factor"
)

plt.title(
    "Confidence Threshold vs Profit Factor"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 18. Confidence vs 平均Return
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    threshold_results[
        "threshold"
    ],
    threshold_results[
        "gross_avg_return"
    ]
    *
    100,
    marker="o",
    label="Cost = 0"
)

plt.plot(
    threshold_results[
        "threshold"
    ],
    threshold_results[
        "net_avg_return"
    ]
    *
    100,
    marker="o",
    label="Cost = 0.004%"
)

plt.axhline(
    0,
    linewidth=1
)

plt.xlabel(
    "Minimum confidence"
)

plt.ylabel(
    "Average return / trade (%)"
)

plt.title(
    "Confidence Threshold vs Average Return"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 19. Confidence vs trade count
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    threshold_results[
        "threshold"
    ],
    threshold_results[
        "trades"
    ],
    marker="o"
)

plt.xlabel(
    "Minimum confidence"
)

plt.ylabel(
    "Number of trades"
)

plt.title(
    "Confidence Threshold vs Trade Count"
)

plt.tight_layout()

plt.show()


# ============================================================
# 20. 保存
# ============================================================

OUTPUT_DIR = (
    Path.cwd()
    /
    "15m_confidence_profitability"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)


oos.to_csv(
    OUTPUT_DIR
    /
    "oos_predictions.csv"
)


band_results.to_csv(
    OUTPUT_DIR
    /
    "confidence_bands.csv",
    index=False
)


threshold_results.to_csv(
    OUTPUT_DIR
    /
    "threshold_results.csv",
    index=False
)


side_results.to_csv(
    OUTPUT_DIR
    /
    "buy_sell_thresholds.csv",
    index=False
)


fold_stability.to_csv(
    OUTPUT_DIR
    /
    "fold_stability.csv",
    index=False
)


bootstrap_results.to_csv(
    OUTPUT_DIR
    /
    "bootstrap_results.csv",
    index=False
)


print()

print(
    "===================================="
)

print(
    "検証完了"
)

print(
    "===================================="
)

print(
    "保存先:"
)

print(
    OUTPUT_DIR.resolve()
)

print()

print(
    "最重要チェック項目:"
)

print(
    "1. Confidenceを上げるほどPFが上がるか"
)

print(
    "2. Cost=0なら黒字なのか"
)

print(
    "3. Cost=0.004%でも黒字になる閾値があるか"
)

print(
    "4. BUYとSELLで違いがあるか"
)

print(
    "5. 複数Foldで再現するか"
)

print(
    "6. Bootstrap 95%CIが0より上になるか"
)